In [ ]:
# Install required libraries (for Colab or first-time use in Jupyter)
!pip install albumentations

In [1]:
import os
import shutil
import cv2
import albumentations as A

# --- Configuration ---
SOURCE_FOLDER = "resized_images"
OUTPUT_FOLDER = "augmented_images_all"
VERSION_TAG = "_V01"

def setup_directories():
    """
    Creates the source folder if it doesn't exist and cleans/creates the output folder.
    """
    # Create the source folder for users to put their images in
    if not os.path.exists(SOURCE_FOLDER):
        os.makedirs(SOURCE_FOLDER)
        print(f"✅ Created source directory: '{SOURCE_FOLDER}'")
        print(f"‼️ Please add your images ending with '{VERSION_TAG}' to this folder and run again.")

    # Clean and create the output directory
    if os.path.exists(OUTPUT_FOLDER):
        shutil.rmtree(OUTPUT_FOLDER)
        print(f"🧹 Removed old output directory: '{OUTPUT_FOLDER}'")
    os.makedirs(OUTPUT_FOLDER)
    print(f"✅ Created empty output directory: '{OUTPUT_FOLDER}'")

def augment_and_rename_images():
    """
    Loads all images ending with _V01, creates 3 augmented versions (V02, V03, V04),
    and saves all 4 versions to the output folder.
    """
    # Define the augmentations to be applied
    h_flip = A.HorizontalFlip(p=1.0)
    v_flip = A.VerticalFlip(p=1.0)

    try:
        all_files = os.listdir(SOURCE_FOLDER)
    except FileNotFoundError:
        print(f"❌ Error: Source folder '{SOURCE_FOLDER}' not found.")
        return

    # Filter for valid image files that follow the naming convention
    source_images = [
        f for f in all_files
        if f.lower().endswith(('.png', '.jpg', '.jpeg')) and VERSION_TAG.lower() in f.lower()
    ]

    if not source_images:
        print(f"❌ No images ending with '{VERSION_TAG}' found in '{SOURCE_FOLDER}'. Nothing to process.")
        return

    print(f"\nFound {len(source_images)} source images. Starting version generation...")

    # Loop through each valid source image
    for filename in source_images:
        file_path = os.path.join(SOURCE_FOLDER, filename)

        # Load the image using OpenCV
        image = cv2.imread(file_path)
        if image is None:
            print(f"-- Skipping '{filename}' as it could not be read.")
            continue

        print(f"  -> Processing: {filename}")

        # --- Filename Parsing Logic ---
        base_name, extension = os.path.splitext(filename)
        # Find the part of the name before the version tag (e.g., "5Marla_GF_FP_011")
        # This handles cases like "_v01" or "_V01"
        try:
            name_prefix = base_name.split(VERSION_TAG)[0]
        except IndexError:
            print(f"-- Skipping '{filename}'. Could not find version tag '{VERSION_TAG}' to create prefix.")
            continue

        # 1. Copy the original V01 image to the output folder
        shutil.copy(file_path, os.path.join(OUTPUT_FOLDER, filename))

        # 2. Create V02 (Horizontal Flip)
        img_v02 = h_flip(image=image)['image']
        v02_filename = f"{name_prefix}_V02{extension}"
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, v02_filename), img_v02)

        # 3. Create V03 (Vertical Flip)
        img_v03 = v_flip(image=image)['image']
        v03_filename = f"{name_prefix}_V03{extension}"
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, v03_filename), img_v03)

        # 4. Create V04 (Vertical + Horizontal Flip)
        # Apply horizontal flip to the already vertically flipped image (img_v03)
        img_v04 = h_flip(image=img_v03)['image']
        v04_filename = f"{name_prefix}_V04{extension}"
        cv2.imwrite(os.path.join(OUTPUT_FOLDER, v04_filename), img_v04)

    print("\n✅ Image version generation complete.")


if __name__ == "__main__":
    setup_directories()
    augment_and_rename_images()

✅ Created empty output directory: 'augmented_images_all'

Found 7 source images. Starting version generation...
  -> Processing: 5Marla_GF_FP_011_V01.png
  -> Processing: 5Marla_GF_FP_012_V01.png
  -> Processing: 5Marla_GF_FP_013_V01.png
  -> Processing: 5Marla_GF_FP_014_V01.png
  -> Processing: 5Marla_GF_FP_015_V01.png
  -> Processing: 5Marla_GF_FP_016_V01.png
  -> Processing: 5Marla_GF_FP_017_V01.png

✅ Image version generation complete.
